# Fraud Detection — Feature Engineering
### IEEE-CIS Fraud Detection Dataset
In this notebook we merge the transaction and identity data, handle missing values, encode categorical variables, and prepare the data for modeling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 50)

In [2]:
transaction = pd.read_csv('../data/train_transaction.csv')
identity = pd.read_csv('../data/train_identity.csv')

print(f"Transactions: {transaction.shape[0]:,} rows, {transaction.shape[1]} columns")
print(f"Identity:     {identity.shape[0]:,} rows, {identity.shape[1]} columns")

Transactions: 590,540 rows, 394 columns
Identity:     144,233 rows, 41 columns


In [3]:
df = transaction.merge(identity, on='TransactionID', how='left')

print(f"Merged shape: {df.shape}")
print(f"Transactions before merge: {transaction.shape[0]:,}")
print(f"Transactions after merge:  {df.shape[0]:,}")

Merged shape: (590540, 434)
Transactions before merge: 590,540
Transactions after merge:  590,540


In [4]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Fill numeric missing values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical missing values with the most frequent value
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"Numeric columns:     {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")

Numeric columns:     403
Categorical columns: 31
Missing values remaining: 0


In [5]:
le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(f"Categorical columns encoded: {len(categorical_cols)}")
print(df[categorical_cols].head())


Categorical columns encoded: 31
   ProductCD  card4  card6  P_emaildomain  R_emaildomain  M1  M2  M3  M4  M5  \
0          4      1      1             16             16   1   1   1   2   0   
1          4      2      1             16             16   1   1   1   0   1   
2          4      3      2             35             16   1   1   1   0   0   
3          4      2      2             53             16   1   1   1   0   1   
4          1      2      1             16             16   1   1   1   0   0   

   M6  M7  M8  M9  id_12  id_15  id_16  id_23  id_27  id_28  id_29  id_30  \
0   1   0   0   1      1      0      0      2      0      0      0     42   
1   1   0   0   1      1      0      0      2      0      0      0     42   
2   0   0   0   0      1      0      0      2      0      0      0     42   
3   0   0   0   1      1      0      0      2      0      0      0     42   
4   0   0   0   1      1      1      1      2      0      1      1      7   

   id_31  id_33  id_34  

In [6]:
# Drop columns that aren't useful for modeling
drop_cols = ['TransactionID', 'TransactionDT']

X = df.drop(columns=drop_cols + ['isFraud'])
y = df['isFraud']

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Fraud cases:    {y.sum():,}")
print(f"Legit cases:    {(y == 0).sum():,}")

Features shape: (590540, 431)
Target shape:   (590540,)
Fraud cases:    20,663
Legit cases:    569,877


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} rows")
print(f"Test set:       {X_test.shape[0]:,} rows")
print(f"Train fraud %:  {y_train.mean()*100:.2f}%")
print(f"Test fraud %:   {y_test.mean()*100:.2f}%")

Training set:   472,432 rows
Test set:       118,108 rows
Train fraud %:  3.50%
Test fraud %:   3.50%


In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - Fraud: {y_train.sum():,} | Legit: {(y_train==0).sum():,}")
print(f"After SMOTE  - Fraud: {y_train_resampled.sum():,} | Legit: {(y_train_resampled==0).sum():,}")

Before SMOTE - Fraud: 16,530 | Legit: 455,902
After SMOTE  - Fraud: 455,902 | Legit: 455,902


In [9]:
import os

os.makedirs('../data/processed', exist_ok=True)

X_train_resampled.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train_resampled.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
